# 📊 Analisis & Rekomendasi by.ficlo

Notebook **analisis data** (bukan pembuat dashboard): kualitas data → laporan penjualan → analisis kategori/harga/momen → **Tanya Rekomendasi** → rekonsiliasi dengan data website → ekspor CSV siap impor DB.

Stack: Python · pandas · scikit-learn · Plotly · Jupyter/Colab (+ FastAPI di `api.py`).
**Jalankan dari atas ke bawah.** Di Colab, sel 2 meminta unggah file `.py`, `faq.csv`, dan Excel.

## 1. Persiapan

In [ ]:
!pip install -q pandas plotly openpyxl scikit-learn

In [ ]:
import os, sys
BUTUH = ["data_loader.py", "analytics.py", "recommender.py", "export_data.py", "faq.csv", "Dataset_Dasboard_DS_byficlo.xlsx"]
hilang = [f for f in BUTUH if not os.path.exists(f)]
if hilang and "google.colab" in sys.modules:
    from google.colab import files
    print("Unggah:", hilang)
    for nama in files.upload(): pass
    hilang = [f for f in BUTUH if not os.path.exists(f)]
assert not hilang, f"File belum ada di {os.getcwd()}: {hilang}"
os.makedirs("data_bersih", exist_ok=True)
print("Siap.")

In [ ]:
import pandas as pd, plotly.express as px
import data_loader as dl, analytics as an, recommender as rc, export_data as ex
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
d = dl.load_all("Dataset_Dasboard_DS_byficlo.xlsx")
print({k: len(d[k]) for k in ["produk","bahan","inventaris","penjualan","pesanan","faq"]})

## 2. Kualitas data
Masalah yang ditemukan otomatis saat pembersihan (ID kosong, selisih rumus Excel, HPP salah, dll.).

In [ ]:
for w in d["dq"]["peringatan"]: print("-", w)
d["dq"]["saran_mapping"]

## 3. Laporan penjualan
Angka sama persis dengan halaman Sales Analytics dan endpoint API.

In [ ]:
pj = d['penjualan']
an.laporan_harian(pj)

In [ ]:
an.laporan_bulanan(pj)

In [ ]:
an.produk_terlaris(pj, 10)

In [ ]:
h = an.laporan_harian(pj)
px.bar(h, x="Tanggal", y=["Pendapatan","Laba_Kotor"], barmode="group", title="Pendapatan & laba kotor harian")

## 4. Analisis untuk rekomendasi buket (budget · tema · momen)

In [ ]:
an.band_harga(d['produk'])   # sebaran produk per rentang harga = acuan budget

In [ ]:
kp = an.kategori_performa(pj, d['produk']); kp

In [ ]:
px.bar(kp, x='Kategori', y=['Pendapatan','Laba_Kotor'], barmode='group', title='Kategori: pendapatan vs laba')

In [ ]:
an.momen_performa(d['pesanan'])

In [ ]:
for t in an.insight_otomatis(d): print('-', t)

## 5. Tanya Rekomendasi
Memakai TF-IDF + cosine similarity (scikit-learn) dan skor gabungan budget/tema/momen/popularitas/margin/stok. **Input kosong ditolak.**

In [ ]:
R = rc.Rekomendator(d)
h = R.rekomendasi("150rb", "Manis (Pastel)", "Wisuda")
display(h.produk[["Nama_Produk","Harga_Jual","Skor","Alasan"]])
print("Pelengkap:", [a["Nama"] for a in h.addon]); print(h.catatan)

In [ ]:
# Uji penolakan input kosong
for kasus in [(None, "Romantis", "Wisuda"), (100000, "", "Wisuda"), (100000, "Romantis", "  "), ("", "", "")]:
    try:
        R.rekomendasi(*kasus); print("TIDAK DITOLAK:", kasus)
    except rc.InputKosongError as e:
        print("Ditolak", kasus, "->", e.errors)

In [ ]:
# Peta rekomendasi teratas: budget x momen (tema Romantis)
budgets = [50_000, 100_000, 150_000, 200_000, 300_000]
pd.DataFrame({m: [R.rekomendasi(b, "Romantis", m, top_n=1).produk.iloc[0]["Nama_Produk"] for b in budgets] for m in rc.MOMEN},
             index=[f"Rp{b:,}" for b in budgets]).T

## 6. Rekonsiliasi dengan data pesanan website
Dashboard membaca Excel, jadi untuk membuktikan angkanya cocok, bandingkan dengan ekspor pesanan website (kolom `tanggal`, `total`, opsional `id_pesanan`).
Di bawah, contoh memakai data yang sengaja dibuat sama persis dengan dashboard. **Ganti dengan file ekspor website asli.**

In [ ]:
import io
harian = an.laporan_harian(pj)
contoh = "tanggal,total\n" + "\n".join(f"{r.Tanggal:%Y-%m-%d},{int(r.Pendapatan)}" for r in harian.itertuples())
web, err = an.baca_data_website(io.StringIO(contoh), "contoh.csv")   # ganti: an.baca_data_website("ekspor_website.csv", "ekspor_website.csv")
rk = an.rekonsiliasi(pj, web); print(rk["ringkas"]); rk["harian"]

## 7. Ekspor data bersih siap impor database

In [ ]:
tabel = ex.export_clean(d, "data_bersih")     # produk, bahan, inventaris, faq (tanpa data keuangan)
for nama, ok, pesan in ex.validasi_impor(tabel): print(("OK " if ok else "GAGAL"), nama, "-", pesan)

In [ ]:
# opsional: bangun database SQLite dari CSV + schema.sql (butuh data_bersih/schema.sql)
if os.path.exists("data_bersih/schema.sql"):
    print(ex.build_sqlite(tabel, "ficlo.db", "data_bersih/schema.sql"))
else:
    print("Unggah data_bersih/schema.sql dulu untuk membuat database.")